# Schelling Segregation Model

Thomas Schelling's 1971 model shows how **mild individual preferences** for similar neighbours
can produce **strong group-level segregation** — even when no one is deeply prejudiced.

Two groups of agents (**X** in blue, **O** in orange) are placed randomly on a grid.
Any agent whose same-type neighbours fall below the **similarity threshold** is *unhappy* and
moves to a random empty cell. This repeats until everyone is satisfied — or the round limit is reached.

The surprising result: even a 30 % threshold produces heavy clustering.

## How to use

1. **Run all cells** (Kernel → Restart & Run All).
2. Use the **▶ Next Round** button to step through the simulation one round at a time.
3. Use **↺ Reset** to start a fresh random grid.
4. Change any value in the *Parameters* cell, re-run it, then click **↺ Reset**.

In [11]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

In [ ]:
GRID_SIZE            = 25    # grid is GRID_SIZE × GRID_SIZE cells
RATIO_X              = 0.45 # fraction of cells that start as X agents
RATIO_O              = 0.45  # fraction of cells that start as O agents
                              # empty fraction = 1 − RATIO_X − RATIO_O
SIMILARITY_THRESHOLD = 0.30  # minimum fraction of same-type neighbours to be happy
MAX_ROUNDS           = 100   # stop after this many rounds even if agents are still unhappy

## Model functions

The logic is identical to the terminal version: grid creation, Moore neighbourhood,
happiness test, and the move step. No display code here — that comes in the next cell.

In [13]:
# ── Grid creation ──────────────────────────────────────────────────────────────

def create_grid():
    """Place X agents, O agents, and empty cells randomly on the grid.

    The grid is a list of rows; each row is a list of cells.
    Each cell holds 'X', 'O', or '.' (empty).
    """
    total   = GRID_SIZE * GRID_SIZE
    n_x     = int(total * RATIO_X)
    n_o     = int(total * RATIO_O)
    n_empty = total - n_x - n_o

    flat = ['X'] * n_x + ['O'] * n_o + ['.'] * n_empty
    random.shuffle(flat)

    # Reshape flat list into GRID_SIZE rows of GRID_SIZE cells each.
    return [flat[r * GRID_SIZE : (r + 1) * GRID_SIZE] for r in range(GRID_SIZE)]


# ── Neighbourhood logic ────────────────────────────────────────────────────────

def get_neighbors(grid, row, col):
    """Return the values of the (up to 8) Moore-neighbourhood cells around (row, col).

    Cells at the grid edge have fewer neighbours — we do not wrap around.
    """
    neighbors = []
    for dr in [-1, 0, 1]:      # row offsets: up, same row, down
        for dc in [-1, 0, 1]:  # column offsets: left, same, right
            if dr == 0 and dc == 0:
                continue        # skip the cell itself
            r, c = row + dr, col + dc
            if 0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE:
                neighbors.append(grid[r][c])
    return neighbors


def is_happy(grid, row, col):
    """Return True if the agent at (row, col) meets the similarity threshold.

    Only occupied neighbours count; empty cells are ignored.
    An agent with no occupied neighbours at all is considered happy.
    """
    agent = grid[row][col]
    if agent == '.':
        return True

    neighbors = get_neighbors(grid, row, col)
    occupied  = [n for n in neighbors if n != '.']

    if not occupied:
        return True  # isolated agent — nothing to be unhappy about

    same = sum(1 for n in occupied if n == agent)
    return (same / len(occupied)) >= SIMILARITY_THRESHOLD


# ── Finding and moving unhappy agents ─────────────────────────────────────────

def find_unhappy_agents(grid):
    """Return a list of (row, col) positions for every unhappy agent."""
    return [
        (r, c)
        for r in range(GRID_SIZE)
        for c in range(GRID_SIZE)
        if grid[r][c] != '.' and not is_happy(grid, r, c)
    ]


def move_agents(grid, unhappy_agents):
    """Move each unhappy agent to a randomly chosen empty cell.

    Agents move to a *random* vacancy — not necessarily one that will make
    them happy. Segregation still emerges, which is the surprising result.
    """
    empty_cells = [
        (r, c)
        for r in range(GRID_SIZE)
        for c in range(GRID_SIZE)
        if grid[r][c] == '.'
    ]
    random.shuffle(unhappy_agents)  # random move order — no group gets priority

    for (r, c) in unhappy_agents:
        if not empty_cells:
            break
        idx       = random.randrange(len(empty_cells))
        tr, tc    = empty_cells[idx]
        grid[tr][tc] = grid[r][c]  # place agent in the new cell
        grid[r][c]   = '.'         # vacate the old cell
        empty_cells[idx] = (r, c)  # old cell is now a vacancy


# ── Measurement ───────────────────────────────────────────────────────────────

def compute_segregation_index(grid):
    """Return the mean fraction of same-type occupied neighbours across all agents.

    0.0 = fully mixed,  1.0 = fully segregated,  ~0.5 = random starting point.
    This is the NetLogo 'percent-similar' metric divided by 100.
    """
    scores = []
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if grid[r][c] == '.':
                continue
            agent     = grid[r][c]
            neighbors = get_neighbors(grid, r, c)
            occupied  = [n for n in neighbors if n != '.']
            if occupied:
                same = sum(1 for n in occupied if n == agent)
                scores.append(same / len(occupied))
    return sum(scores) / len(scores) if scores else 0.0

## Visualisation

Instead of ASCII characters, the notebook renders the grid as a colour map:

- 🟦 **Blue** — X agent (happy)
- 🟧 **Orange** — O agent (happy)
- ⬜ **Light grey** — empty cell
- **°** marker — agent is *unhappy* and will move next round

In [14]:
def show_grid(grid, unhappy_set, round_num, n_agents):
    """Render the current grid as a coloured matplotlib figure."""
    n_unhappy   = len(unhappy_set)
    pct_unhappy = n_unhappy / n_agents * 100 if n_agents > 0 else 0
    seg         = compute_segregation_index(grid)

    # Build a 2-D numeric array for imshow: 0 = empty, 1 = X, 2 = O.
    data = []
    for r in range(GRID_SIZE):
        row_data = []
        for c in range(GRID_SIZE):
            if grid[r][c] == '.':
                row_data.append(0)
            elif grid[r][c] == 'X':
                row_data.append(1)
            else:
                row_data.append(2)
        data.append(row_data)

    # Three-colour map: light grey / blue / orange.
    cmap = ListedColormap(['#eeeeee', '#4472C4', '#ED7D31'])

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')

    # Overlay ° on every unhappy agent so they stand out.
    for (r, c) in unhappy_set:
        ax.text(c, r, '°', ha='center', va='center',
                fontsize=9, color='white', fontweight='bold')

    # Thin white lines to separate cells.
    for i in range(GRID_SIZE + 1):
        ax.axhline(i - 0.5, color='white', linewidth=0.4)
        ax.axvline(i - 0.5, color='white', linewidth=0.4)

    ax.set_xticks([])
    ax.set_yticks([])

    # Legend.
    legend_patches = [
        mpatches.Patch(color='#4472C4', label='X agent (happy)'),
        mpatches.Patch(color='#ED7D31', label='O agent (happy)'),
        mpatches.Patch(color='#eeeeee', label='empty cell'),
    ]
    ax.legend(handles=legend_patches, loc='upper left',
              bbox_to_anchor=(1.02, 1), framealpha=0.9, fontsize=9)

    ax.set_title(
        f'Round {round_num}\n'
        f'Unhappy: {n_unhappy} / {n_agents}  ({pct_unhappy:.1f}%)     '
        f'Segregation index: {seg:.2f}',
        fontsize=11
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)  # release memory — important inside a button loop

## Run the simulation

Click **▶ Next Round** to advance one step at a time.  
Click **↺ Reset** to start over with a new random grid.

In [ ]:
# ── Simulation state ───────────────────────────────────────────────────────────
# We store the grid and counters in a dict so that button callbacks
# (which are separate functions) can all read and update the same state.

state = {'grid': None, 'round': 0, 'done': False, 'n_agents': 0}

out = widgets.Output()  # output area where the grid will be drawn


def display_current_state():
    """Redraw the grid and stats inside the output widget."""
    unhappy_agents = find_unhappy_agents(state['grid'])
    unhappy_set    = set(unhappy_agents)

    with out:
        clear_output(wait=True)
        show_grid(state['grid'], unhappy_set, state['round'], state['n_agents'])

        if state['done']:
            if not unhappy_agents:
                print('✓  All agents are happy — simulation complete.')
            else:
                print(f'Reached the {MAX_ROUNDS}-round limit. Stopping.')


def on_next_round(btn):
    """Button callback: move unhappy agents one round forward, then redisplay."""
    if state['done']:
        return

    unhappy_agents = find_unhappy_agents(state['grid'])

    # If no one is unhappy before we move, the model has already converged.
    if not unhappy_agents:
        state['done'] = True
        display_current_state()
        return

    move_agents(state['grid'], unhappy_agents)
    state['round'] += 1

    if state['round'] >= MAX_ROUNDS:
        state['done'] = True

    display_current_state()


def on_reset(btn):
    """Button callback: build a fresh random grid and reset the round counter."""
    state['grid']     = create_grid()
    state['round']    = 0
    state['done']     = False
    state['n_agents'] = sum(
        1 for r in range(GRID_SIZE)
          for c in range(GRID_SIZE)
          if state['grid'][r][c] != '.'
    )
    display_current_state()


# ── Buttons ────────────────────────────────────────────────────────────────────
btn_next  = widgets.Button(
    description='▶  Next Round',
    button_style='primary',
    layout=widgets.Layout(width='140px')
)
btn_reset = widgets.Button(
    description='↺  Reset',
    button_style='warning',
    layout=widgets.Layout(width='110px')
)

btn_next.on_click(on_next_round)
btn_reset.on_click(on_reset)

# Draw the initial grid automatically when this cell runs.
on_reset(None)

display(widgets.HBox([btn_next, btn_reset]), out)

Output()